In [16]:
Q: 1
# Question : Building a SQLite Database with Python and Pandas Integration
# Problem Statement
# You are a data engineer at a small online bookstore. You need to build a database system to manage books, customers, and orders. Currently, data is scattered across CSV files, causing data integrity issues and making it difficult to track relationships. Your task is to design a relational database schema, create tables with proper relationships, populate them with data, query the database, and integrate with pandas for analysis.

# Task 1: Database Design and Table Creation (35 points)
# Design and create a SQLite database named bookstore.db with the following tables:

# 1. Books Table:

# book_id (INTEGER, Primary Key, Auto-increment)
# title (TEXT, Not Null)
# author (TEXT, Not Null)
# price (REAL, Not Null)
# stock_quantity (INTEGER, Default 0)
# 2. Customers Table:

# customer_id (INTEGER, Primary Key, Auto-increment)
# name (TEXT, Not Null)
# email (TEXT, Unique, Not Null)
# city (TEXT)
# join_date (TEXT)
# 3. Orders Table:

# order_id (INTEGER, Primary Key, Auto-increment)
# customer_id (INTEGER, Foreign Key referencing Customers)
# book_id (INTEGER, Foreign Key referencing Books)
# quantity (INTEGER, Not Null)
# order_date (TEXT, Not Null)
# total_amount (REAL)
# Your tasks:

# Create the database connection
# Write SQL CREATE TABLE statements with appropriate constraints
# Create all three tables
# Display the schema of each table using PRAGMA commands

import sqlite3
import os
import pandas as pd # Import pandas

# Remove the database file if it already exists for a clean run
if os.path.exists("bookstore.db"):
    os.remove("bookstore.db")

# 1️ Create database connection
connect = sqlite3.connect("bookstore.db")
cursor = connect.cursor()

# 2️ Create Books table
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER DEFAULT 0
)
""")

# 3️ Create Customers table
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    city TEXT,
    join_date TEXT
)
""")

# 4️ Create Orders table
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    book_id INTEGER,
    quantity INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id)
)
""")

connect.commit()

# 5️ Display schema using PRAGMA

print("Books Table Schema")
for row in cursor.execute("PRAGMA table_info(books)"):
    print(row)

print("\nCustomers Table Schema")
for row in cursor.execute("PRAGMA table_info(customers)"):
    print(row)

print("\nOrders Table Schema")
for row in cursor.execute("PRAGMA table_info(orders)"):
    print(row)

# connect.close()  <-- Moved this to the end of the script

# Task 2: Data Insertion and Querying (35 points)
# Part A: Insert Data

# Insert the following data into your tables:

# Books:

books_data = [
    ('Python Programming', 'John Smith', 599.99, 25),
    ('Data Science Handbook', 'Jane Doe', 899.50, 15),
    ('Machine Learning Basics', 'Alan Turing', 1299.00, 10),
    ('SQL Essentials', 'Edgar Codd', 499.99, 30),
    ('Web Development', 'Tim Berners', 799.00, 20)
]
# Customers:

customers_data = [
    ('Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
    ('Priya Patel', 'priya@email.com', 'Delhi', '2024-01-20'),
    ('Amit Kumar', 'amit@email.com', 'Bangalore', '2024-02-01'),
    ('Sneha Reddy', 'sneha@email.com', 'Hyderabad', '2024-02-10'),
    ('Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')
]
# Orders:

orders_data = [
    (1, 1, 2, '2024-03-01', 1199.00),
    (1, 2, 1, '2024-03-02', 899.50),
    (2, 1, 1, '2024-03-03', 599.99),
    (2, 3, 1, '2024-03-05', 1299.00),
    (3, 4, 3, '2024-03-07', 1499.97),
    (4, 2, 1, '2024-03-10', 899.50),
    (5, 5, 2, '2024-03-12', 1598.00)
]
# Use executemany() to insert all records efficiently
# Commit the changes
# Query and display all records from each table to verify insertion
# Part B: Complex Queries

# Write SQL queries to:

# Find all customers from Mumbai
# Find books with price > 800 and stock > 10
# Count total number of orders
# Find the customer who has placed the most orders (use GROUP BY and ORDER BY)
# Calculate total revenue (sum of all order amounts)

# Insert Books
cursor.executemany("""
INSERT INTO books (title, author, price, stock_quantity)
VALUES (?, ?, ?, ?)
""", books_data)

# Insert Customers
cursor.executemany("""
INSERT INTO customers (name, email, city, join_date)
VALUES (?, ?, ?, ?)
""", customers_data)

# Insert Orders
cursor.executemany("""
INSERT INTO orders (customer_id, book_id, quantity, order_date, total_amount)
VALUES (?, ?, ?, ?, ?)
""", orders_data)

connect.commit()

# Query and display all records from each table to verify insertion
print("\n--- All Books ---")
for row in cursor.execute("SELECT * FROM books"):
    print(row)

print("\n--- All Customers ---")
for row in cursor.execute("SELECT * FROM customers"):
    print(row)

print("\n--- All Orders ---")
for row in cursor.execute("SELECT * FROM orders"):
    print(row)

# Part B - Complex Queries
# 1️⃣ Find all customers from Mumbai
query1 = "SELECT * FROM customers WHERE city = 'Mumbai'"
for row in cursor.execute(query1):
    print(row)
# 2️⃣ Find books with price > 800 and stock > 10
query2 = """
SELECT * FROM books
WHERE price > 800 AND stock_quantity > 10
"""

for row in cursor.execute(query2):
    print(row)
# 3️⃣ Count total number of orders
query3 = "SELECT COUNT(*) FROM orders"

cursor.execute(query3)
print("Total Orders:", cursor.fetchone()[0])
# 4️⃣ Customer who placed the most orders
query4 = """
SELECT customer_id, COUNT(order_id) as total_orders
FROM orders
GROUP BY customer_id
ORDER BY total_orders DESC
LIMIT 1
"""

cursor.execute(query4)
print(cursor.fetchone())

# Better version (with customer name):

query4 = """
SELECT customers.name, COUNT(orders.order_id) as total_orders
FROM orders
JOIN customers
ON orders.customer_id = customers.customer_id
GROUP BY customers.name
ORDER BY total_orders DESC
LIMIT 1
"""

cursor.execute(query4)
print(cursor.fetchone())
# 5️⃣ Calculate Total Revenue
query5 = "SELECT SUM(total_amount) FROM orders"

cursor.execute(query5)
print("Total Revenue:", cursor.fetchone()[0])


# TASK 3: PANDAS INTEGRATION
books_df = pd.read_sql("SELECT * FROM books", connect)
customers_df = pd.read_sql("SELECT * FROM customers", connect)
orders_df = pd.read_sql("SELECT * FROM orders", connect)

print("\nBooks DataFrame")
print(books_df.head(3))
print("\nCustomers DataFrame")
print(customers_df.head(3))
print("\nOrders DataFrame")
print(orders_df.head(3))

# Merge data
orders_customers = pd.merge(orders_df, customers_df, on="customer_id")
full_report = pd.merge(orders_customers, books_df, on="book_id")
order_report = full_report[["order_id", "name", "city", "title", "quantity", "total_amount"]]
print("\nORDER REPORT")
print(order_report)

# Analysis
avg_order_value = orders_df["total_amount"].mean()
print("\nAverage Order Value:", avg_order_value)
orders_by_city = full_report.groupby("city")["order_id"].count()
print("\nOrders by City")
print(orders_by_city)
popular_book = full_report.groupby("title")["quantity"].sum().sort_values(ascending=False)
print("\nMost Popular Book")
print(popular_book.head(1))

# PANDAS TO SQL
discounts_data = { 'book_id': [1,2,3,4,5], 'discount_percent': [10,15,5,20,12] }
discounts_df = pd.DataFrame(discounts_data)
discounts_df.to_sql("discounts", connect, if_exists="replace", index=False)
print("\nDISCOUNTS TABLE")
print(pd.read_sql("SELECT * FROM discounts", connect))

# Discount query
query = """
SELECT books.title, books.price, discounts.discount_percent, price - (price * discount_percent / 100) AS discounted_price
FROM books
JOIN discounts ON books.book_id = discounts.book_id
"""
discounted_books = pd.read_sql(query, connect)
print("\nBOOKS WITH DISCOUNTED PRICE")
print(discounted_books)

connect.close()

Books Table Schema
(0, 'book_id', 'INTEGER', 0, None, 1)
(1, 'title', 'TEXT', 1, None, 0)
(2, 'author', 'TEXT', 1, None, 0)
(3, 'price', 'REAL', 1, None, 0)
(4, 'stock_quantity', 'INTEGER', 0, '0', 0)

Customers Table Schema
(0, 'customer_id', 'INTEGER', 0, None, 1)
(1, 'name', 'TEXT', 1, None, 0)
(2, 'email', 'TEXT', 1, None, 0)
(3, 'city', 'TEXT', 0, None, 0)
(4, 'join_date', 'TEXT', 0, None, 0)

Orders Table Schema
(0, 'order_id', 'INTEGER', 0, None, 1)
(1, 'customer_id', 'INTEGER', 0, None, 0)
(2, 'book_id', 'INTEGER', 0, None, 0)
(3, 'quantity', 'INTEGER', 1, None, 0)
(4, 'order_date', 'TEXT', 1, None, 0)
(5, 'total_amount', 'REAL', 0, None, 0)

--- All Books ---
(1, 'Python Programming', 'John Smith', 599.99, 25)
(2, 'Data Science Handbook', 'Jane Doe', 899.5, 15)
(3, 'Machine Learning Basics', 'Alan Turing', 1299.0, 10)
(4, 'SQL Essentials', 'Edgar Codd', 499.99, 30)
(5, 'Web Development', 'Tim Berners', 799.0, 20)

--- All Customers ---
(1, 'Rahul Sharma', 'rahul@email.com', 'M